<a href="https://colab.research.google.com/github/jnolorbe/hydrology-peru/blob/main/notebooks/05_analisis_por_eventos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Análisis por eventos de ENOS (antes *script* 09)

Respuesta del agua total almacenada (TWS, GRACE/GRACE-FO JPL RL06.3Mv04) a cinco eventos de impacto conocido en
el Perú, incluidos los que la clasificación por regímenes no asigna: El Niño 2009–2010, La Niña 2010–2012,
El Niño 2015–2016 y los eventos costeros de 2017 y 2023–2024 (tesis, Secciones 3.6.9 y 4.4.4).

**Ventanas.** Se definen con los índices y no con el TWS, para que la elección de los extremos no determine el
signo del cambio: bloque continuo con ONI ≥ +0.5 (2009–2010 y 2015–2016), ONI ≤ −0.5 (La Niña 2010–2012; se unen
interrupciones de hasta tres meses) e ICEN ≥ +0.4 (eventos costeros, criterio del ENFEN).

**Métrica.** Con la serie desestacionalizada de cada ámbito (como en el cuaderno 02),
ΔS = media de los tres últimos meses de la ventana − media de los tres meses previos al inicio, en cm y en Gt
(1 cm sobre 1 km² = 10⁻⁵ Gt). Como referencia se da el percentil de ΔS entre todas las ventanas de igual duración
de 2002–2025: es descriptivo, no una prueba de hipótesis, porque las ventanas se solapan y la serie está
autocorrelacionada. Se repite sin tendencia lineal y con ventanas definidas con los índices E y C.

# 1. Entorno y datos

In [1]:
!pip install --quiet netCDF4 xarray rioxarray geopandas shapely statsmodels

import os
import numpy as np
import pandas as pd

REPO_NAME = "hydrology-peru"
GITHUB_USERNAME = "jnolorbe"  # cambiar si se usa una copia propia
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    %cd {REPO_NAME}
    !git pull
    %cd ..

In [2]:
# ── Funciones comunes del repositorio (notebooks/comun.py) ─────────────────────
import sys, os, importlib
sys.path.insert(0, os.path.join(REPO_NAME, 'notebooks'))
import comun; importlib.reload(comun)
from comun import *
DERIV = os.path.join(REPO_NAME, 'data', 'processed_derived')
os.makedirs(DERIV, exist_ok=True)
pd.set_option('display.width', 200, 'display.max_columns', 30)
verificar(REPO_NAME)

Verificación correcta: 288 meses, 14 ámbitos, El Niño E 13 meses en 5 episodios, 69 registros in situ.


# 2. Ventanas de los eventos y cambio del almacenamiento

In [3]:
ev = analisis_eventos(REPO_NAME)
ev['ventanas'].to_csv(os.path.join(DERIV, 'event_windows.csv'), index=False)
ev['cambios'].to_csv(os.path.join(DERIV, 'event_change.csv'), index=False)
ev['sensibilidad'].to_csv(os.path.join(DERIV, 'event_sensitivity_EC.csv'), index=False)
display(ev['ventanas'].round(2))

,evento,inicio,fin,meses,imputados,E_max,E_min,C_max,C_min
0,El Niño 2009–10,2009-08,2010-03,8,0,0.67,-0.10,1.82,0.53
1,La Niña 2010–12,2010-06,2012-02,21,3,0.84,-0.76,-0.56,-2.19
2,El Niño 2015–16,2014-10,2016-04,19,6,2.40,-0.96,1.96,0.61
3,El Niño costero 2017,2016-12,2017-04,5,1,0.94,-0.12,-0.01,-0.41
4,El Niño costero 2023–24,2023-02,2024-02,13,0,2.53,0.39,2.00,-1.11


In [4]:
cambios = ev['cambios']
tabla_cm = cambios[cambios.variante == 'base'].pivot(index='ambito', columns='evento', values='dS_cm').loc[AMBITOS, ev['eventos']]
tabla_pc = cambios[cambios.variante == 'base'].pivot(index='ambito', columns='evento', values='percentil').loc[AMBITOS, ev['eventos']]
print('ΔS por ámbito (cm)'); display(tabla_cm.rename(index=NOMBRE_ES).round(1))
print('Percentil entre ventanas de igual duración'); display(tabla_pc.rename(index=NOMBRE_ES).round(0))
print('ΔS total de los 14 ámbitos (Gt)')
display(cambios.groupby(['variante', 'evento'], sort=False).dS_Gt.sum().round(1).unstack()[ev['eventos']])

ΔS por ámbito (cm)


evento,El Niño 2009–10,La Niña 2010–12,El Niño 2015–16,El Niño costero 2017,El Niño costero 2023–24
ambito,,,,,
Jequetepeque-Zarumilla,-0.4,0.2,1.4,2.2,4.0
Huarmey-Chicama,-7.9,2.1,-0.7,0.2,4.5
Marañón,-7.4,3.4,-1.2,2.9,5.7
Huallaga,-10.0,5.5,-4.0,4.8,5.7
Amazonas,-10.7,10.2,-6.5,5.6,8.0
Cañete-Fortaleza,-1.3,1.9,-1.1,-0.7,1.6
Mantaro,-2.5,4.6,-2.3,-1.5,5.0
Cháparra-Chincha,-0.7,4.7,-2.1,-1.2,1.4
Ucayali,-8.6,8.7,-8.3,-0.8,7.1


Percentil entre ventanas de igual duración


evento,El Niño 2009–10,La Niña 2010–12,El Niño 2015–16,El Niño costero 2017,El Niño costero 2023–24
ambito,,,,,
Jequetepeque-Zarumilla,34.0,52.0,79.0,97.0,100.0
Huarmey-Chicama,0.0,73.0,35.0,53.0,96.0
Marañón,0.0,86.0,28.0,89.0,99.0
Huallaga,1.0,93.0,19.0,95.0,95.0
Amazonas,1.0,97.0,18.0,91.0,95.0
Cañete-Fortaleza,4.0,95.0,12.0,17.0,97.0
Mantaro,10.0,94.0,19.0,21.0,99.0
Cháparra-Chincha,28.0,99.0,12.0,13.0,82.0
Ucayali,4.0,96.0,8.0,47.0,93.0


ΔS total de los 14 ámbitos (Gt)


evento,El Niño 2009–10,La Niña 2010–12,El Niño 2015–16,El Niño costero 2017,El Niño costero 2023–24
variante,,,,,
base,-74.2,74.2,-57.6,4.4,63.7
sin_tendencia,-73.7,75.6,-56.3,4.8,64.6


# 3. Sensibilidad a la definición de las ventanas (índices E y C)

In [5]:
display(ev['sensibilidad'].round(2))

,variante,evento,ventana,r,mismo_signo
0,EC_0.5,El Niño 2009–10,2009-08/2010-04,0.99,14.0
1,EC_0.5,La Niña 2010–12,2010-06/2012-06,0.87,14.0
2,EC_0.5,El Niño 2015–16,2014-10/2016-06,0.87,14.0
3,EC_0.5,El Niño costero 2017,2017-02/2017-04,0.98,14.0
4,EC_0.5,El Niño costero 2023–24,2023-03/2024-02,0.98,14.0
5,EC_regimen,El Niño 2009–10,2009-08/2010-04,0.99,14.0
6,EC_regimen,La Niña 2010–12,2010-08/2012-05,0.87,14.0
7,EC_regimen,El Niño 2015–16,2015-06/2016-03,0.88,12.0
8,EC_regimen,El Niño costero 2017,no detectado,NaN,NaN
9,EC_regimen,El Niño costero 2023–24,2023-03/2024-01,0.98,14.0


### 📊 Lectura de los resultados

Los eventos costeros de 2017 y 2023–2024 aumentan el almacenamiento del norte, mientras que El Niño 2009–2010 y
2015–2016 lo reducen en el norte andino-amazónico y la selva, y La Niña 2010–2012 lo aumenta. En 2017, fuera del
régimen El Niño E, el norte gana y la sierra sur pierde almacenamiento con cambios en los extremos de su
distribución, que no varían al eliminar la tendencia. El análisis es descriptivo: los eventos se eligieron por su
impacto conocido y los percentiles no son pruebas de hipótesis (tesis, Sección 4.4.4).